# 4_Cálculo del valor de decisión óptimo para el caso de una regla de decisión ML en observación discreta


## Consigna

El modelo de generación de ruido más común en comunicaciones digitales que resulta en una observación de valores discretos es el **canal simétrico binario**, del inglés **Binary Symmetric Channel** o **BSC**.

La observación ruidosa se diferencia del símbolo de entrada binaria con probabilidad $p$, llamada probabilidad de error o probabilidad de cruce del BSC.

Se transmite un símbolo binario

$$
A \in \{0,1\}
$$

con probabilidades a priori

$$
P_A(0)=q, \qquad P_A(1)=1-q.
$$

La observación $Y\in\{0,1\}$ también es binaria e igual a $A$ con probabilidad $1-p$.

1. Encuentre cualitativamente la regla de decisión ML para $q=0.6$ y $p=0.3$.
2. Encuentre cualitativamente la regla de decisión ML para $q=0.5$ y $p=0.3$.
3. Determine si esta regla de decisión es óptima en el punto 1 o en el punto 2, es decir, si produce la mínima probabilidad de error.
4. Proponga valores de $p$ y $q$ para que la regla de decisión ML sea óptima.


## Modelo BSC

El BSC conserva el bit con probabilidad $1-p$ y lo invierte con probabilidad $p$.

| Observación $y$ | $P_{Y\mid A}(y\mid 0)$ | $P_{Y\mid A}(y\mid 1)$ |
|---:|---:|---:|
| $0$ | $1-p$ | $p$ |
| $1$ | $p$ | $1-p$ |

Estas probabilidades condicionales son las **verosimilitudes**. La regla ML decide usando solo esas verosimilitudes, por eso **ML no usa $q$**.


In [ ]:
import numpy as np
import matplotlib.pyplot as plt

plt.style.use("seaborn-v0_8-whitegrid")
rng = np.random.default_rng(20260508)


In [ ]:
def bsc_likelihoods(p):
    y_values = np.array([0, 1])
    py_given_a0 = np.array([1 - p, p])
    py_given_a1 = np.array([p, 1 - p])
    return y_values, py_given_a0, py_given_a1

def ml_decisions(p):
    if p < 0.5:
        return np.array([0, 1])
    if p > 0.5:
        return np.array([1, 0])
    return np.array([-1, -1])

def map_decisions(q, p):
    score_y0_a0 = q * (1 - p)
    score_y0_a1 = (1 - q) * p
    score_y1_a0 = q * p
    score_y1_a1 = (1 - q) * (1 - p)

    decision_y0 = 0 if score_y0_a0 >= score_y0_a1 else 1
    decision_y1 = 0 if score_y1_a0 >= score_y1_a1 else 1
    return np.array([decision_y0, decision_y1])

def exact_error_probability(q, p, decisions):
    joint = {
        (0, 0): q * (1 - p),
        (0, 1): q * p,
        (1, 0): (1 - q) * p,
        (1, 1): (1 - q) * (1 - p),
    }
    pe = 0.0
    for (a, y), prob in joint.items():
        if decisions[y] != a:
            pe += prob
    return pe

def simulate_bsc(q, p, decisions, n_samples=300000, seed=1234):
    local_rng = np.random.default_rng(seed)
    a = np.where(local_rng.random(n_samples) < q, 0, 1)
    flips = local_rng.random(n_samples) < p
    y = np.bitwise_xor(a, flips.astype(int))
    a_hat = decisions[y]
    return np.mean(a_hat != a)


## Regla ML general

La regla de máxima verosimilitud elige el valor de $A$ que hace más probable la observación recibida $Y=y$:

$$
\hat A_{ML}(y)=\arg\max_{a\in\{0,1\}} P_{Y\mid A}(y\mid a).
$$

Como el problema es binario, para cada observación comparamos:

$$
P_{Y\mid A}(y\mid 0)
\quad \text{contra} \quad
P_{Y\mid A}(y\mid 1).
$$

### Si se observa $y=0$

$$
P_{Y\mid A}(0\mid 0)=1-p, \qquad P_{Y\mid A}(0\mid 1)=p.
$$

Si $p<0.5$, entonces $1-p>p$, por lo tanto:

$$
\hat A_{ML}(0)=0.
$$

### Si se observa $y=1$

$$
P_{Y\mid A}(1\mid 0)=p, \qquad P_{Y\mid A}(1\mid 1)=1-p.
$$

Si $p<0.5$, entonces $p<1-p$, por lo tanto:

$$
\hat A_{ML}(1)=1.
$$

Conclusión para $p<0.5$:

$$
\boxed{\hat A_{ML}=Y}
$$


## 1. Caso $q=0.6$ y $p=0.3$

Datos:

$$
q=0.6, \qquad p=0.3, \qquad 1-p=0.7.
$$

La regla ML no usa $q$, así que solo miramos las verosimilitudes.

### Si se observa $y=0$

$$
P_{Y\mid A}(0\mid 0)=0.7, \qquad P_{Y\mid A}(0\mid 1)=0.3.
$$

Como $0.7>0.3$:

$$
\boxed{\hat A_{ML}(0)=0}
$$

### Si se observa $y=1$

$$
P_{Y\mid A}(1\mid 0)=0.3, \qquad P_{Y\mid A}(1\mid 1)=0.7.
$$

Como $0.7>0.3$:

$$
\boxed{\hat A_{ML}(1)=1}
$$

Por lo tanto:

$$
\boxed{\hat A_{ML}=Y}
$$

El valor $q=0.6$ no cambia la regla ML, porque $q$ es una probabilidad a priori y ML no la incorpora.


## 2. Caso $q=0.5$ y $p=0.3$

Datos:

$$
q=0.5, \qquad p=0.3, \qquad 1-p=0.7.
$$

Como $p=0.3<0.5$, el canal es más probable que conserve el bit a que lo invierta.

### Si se observa $y=0$

$$
P_{Y\mid A}(0\mid 0)=0.7>0.3=P_{Y\mid A}(0\mid 1),
$$

entonces:

$$
\boxed{\hat A_{ML}(0)=0}.
$$

### Si se observa $y=1$

$$
P_{Y\mid A}(1\mid 1)=0.7>0.3=P_{Y\mid A}(1\mid 0),
$$

entonces:

$$
\boxed{\hat A_{ML}(1)=1}.
$$

Por lo tanto:

$$
\boxed{\hat A_{ML}=Y}
$$


In [ ]:
p = 0.3
y_values, py_given_a0, py_given_a1 = bsc_likelihoods(p)

width = 0.35
x = np.arange(len(y_values))

plt.figure(figsize=(8, 4))
plt.bar(x - width / 2, py_given_a0, width, label=r"$P_{Y\mid A}(y\mid 0)$")
plt.bar(x + width / 2, py_given_a1, width, label=r"$P_{Y\mid A}(y\mid 1)$")
plt.xticks(x, y_values)
plt.xlabel("Observación y")
plt.ylabel("Verosimilitud")
plt.title("Verosimilitudes del BSC para p=0.3")
plt.legend()
plt.show()


## 3. ¿La regla ML es óptima en los puntos 1 y 2?

Para minimizar la probabilidad de error, la regla óptima es MAP:

$$
\hat A_{MAP}(y)=\arg\max_{a\in\{0,1\}} P_{A\mid Y}(a\mid y).
$$

Usando Bayes, como $P_Y(y)$ no depende de $a$, MAP compara:

$$
P_A(0)P_{Y\mid A}(y\mid 0)
\quad \text{contra} \quad
P_A(1)P_{Y\mid A}(y\mid 1).
$$

ML será óptima si sus decisiones coinciden con MAP.


### Punto 1: $q=0.6$ y $p=0.3$

Para $y=0$:

$$
P_A(0)P_{Y\mid A}(0\mid 0)=0.6\cdot0.7=0.42
$$

$$
P_A(1)P_{Y\mid A}(0\mid 1)=0.4\cdot0.3=0.12
$$

MAP decide $0$, igual que ML.

Para $y=1$:

$$
P_A(0)P_{Y\mid A}(1\mid 0)=0.6\cdot0.3=0.18
$$

$$
P_A(1)P_{Y\mid A}(1\mid 1)=0.4\cdot0.7=0.28
$$

MAP decide $1$, igual que ML.

Entonces:

$$
\boxed{\hat A_{ML}=\hat A_{MAP}=Y}
$$

Por lo tanto, para $q=0.6$ y $p=0.3$, la regla ML **sí es óptima**.


### Punto 2: $q=0.5$ y $p=0.3$

Cuando $q=0.5$:

$$
P_A(0)=P_A(1)=0.5.
$$

Si las probabilidades a priori son iguales, MAP se reduce a ML porque ambos lados de la comparación MAP quedan multiplicados por el mismo factor $0.5$.

Entonces:

$$
\boxed{\hat A_{ML}=\hat A_{MAP}=Y}
$$

Por lo tanto, para $q=0.5$ y $p=0.3$, la regla ML **sí es óptima**.


## Probabilidad de error en los dos casos

En ambos casos se usa la regla

$$
\hat A=Y.
$$

Con esa regla, el receptor se equivoca exactamente cuando el BSC invierte el bit. Por lo tanto:

$$
P_e=p.
$$

Para $p=0.3$:

$$
\boxed{P_e=0.3}
$$


## 4. Proponer valores de $p$ y $q$ para que ML sea óptima

Una forma directa de garantizar que ML sea óptima es elegir hipótesis equiprobables:

$$
q=0.5.
$$

Por ejemplo:

$$
\boxed{p=0.3, \qquad q=0.5}
$$

Con esos valores, MAP y ML coinciden.

Más generalmente, para el caso $p<0.5$, ML decide $\hat A=Y$. Para que esa regla también sea MAP, deben cumplirse estas dos condiciones:

$$
q(1-p)\ge(1-q)p
$$

y

$$
(1-q)(1-p)\ge qp.
$$

Despejando:

$$
\boxed{p\le q\le 1-p}
$$

Por eso, si $p=0.3$, cualquier $q$ entre $0.3$ y $0.7$ hace que ML sea óptima.


## Verificación por Monte Carlo

Esta simulación no es necesaria para resolver el ejercicio, pero ayuda a comprobar que las cuentas cierran.


In [ ]:
for idx, q in enumerate([0.6, 0.5], start=1):
    p = 0.3
    decisions_ml = ml_decisions(p)
    decisions_map = map_decisions(q, p)
    pe_exact = exact_error_probability(q, p, decisions_ml)
    pe_mc = simulate_bsc(q, p, decisions_ml, seed=20260508 + idx)

    print(f"Caso {idx}: q={q:.1f}, p={p:.1f}")
    print(f"ML:  y=0 -> A_hat={decisions_ml[0]}, y=1 -> A_hat={decisions_ml[1]}")
    print(f"MAP: y=0 -> A_hat={decisions_map[0]}, y=1 -> A_hat={decisions_map[1]}")
    print(f"ML coincide con MAP: {np.array_equal(decisions_ml, decisions_map)}")
    print(f"Pe exacta con ML: {pe_exact:.4f}")
    print(f"Pe Monte Carlo:   {pe_mc:.4f}")
    print()


## Respuesta final

Para $q=0.6$ y $p=0.3$:

$$
\boxed{\hat A_{ML}=Y}
$$

Para $q=0.5$ y $p=0.3$:

$$
\boxed{\hat A_{ML}=Y}
$$

En ambos casos dados, ML coincide con MAP, por lo tanto la regla ML es óptima y produce la mínima probabilidad de error.

La probabilidad de error es:

$$
\boxed{P_e=0.3}
$$

Un par válido para que ML sea óptima es:

$$
\boxed{p=0.3, \qquad q=0.5}
$$
